# BenchJack sandbox hardening — 0.2.0 vs HEAD

A guided walkthrough of the same mechanism `run_comparison.py` automates: plant a BenchJack-style `conftest.py` exploit against a three-test fail-by-design task, run the same attack through benchflow's `SDK().run()` twice — once under `benchflow==0.2.0` from PyPI and once under the current HEAD — and compare rewards.

**Prerequisites.** This notebook assumes you have already run `python labs/benchjack-sandbox-hardening/run_comparison.py` at least once. That creates the two pinned venvs under `.venvs/bf-0.2.0/` and `.venvs/bf-head/` that the cells below reuse.

**Baking outputs.** The committed notebook has empty output cells on purpose — rendering baked outputs on GitHub requires re-executing against Docker, which only makes sense immediately before committing. To rebuild:

```sh
jupyter nbconvert --to notebook --execute --inplace \
    labs/benchjack-sandbox-hardening/comparison.ipynb
```

In [ ]:
# The three designed-to-fail tests and the pytest exploit payload we'll
# plant against them. Note: nothing about these tests is load-bearing —
# they exist solely so the reward flips from 0.0 (honest run) to 1.0
# (exploit run) in a way that's impossible to achieve legitimately.

print('=' * 72)
print('pattern1_conftest_hook/tests/test_outputs.py')
print('=' * 72)
with open('pattern1_conftest_hook/tests/test_outputs.py') as f:
    print(f.read())

print('=' * 72)
print('pattern1_conftest_hook/environment/conftest_exploit.py')
print('=' * 72)
with open('pattern1_conftest_hook/environment/conftest_exploit.py') as f:
    print(f.read())

## How the hook works

`pytest_collection_modifyitems` fires once, right after pytest discovers test functions but before any of them runs. We mutate `item.obj` on every collected test — that's the attribute pytest reads when it calls the test function. Replacing it with `lambda *args, **kwargs: None` means pytest invokes a no-op for every test, records the outcome as "passed," and exits 0. None of the original asserts ever execute.

The file has to be *named* `conftest.py` and has to land somewhere pytest's collection walk visits. The whole point of this demo is that 0.2.0 lets both conditions happen and 0.2.1 breaks them in four independent ways.

In [ ]:
# The "attacker": a shell script that runs as benchflow's `oracle` agent
# inside the task container. Oracle is a built-in special case in
# SDK.run() that executes solution/solve.sh instead of going through an
# ACP agent loop. We repurpose it as a deterministic attacker:
# the "solution" it runs IS the exploit payload.

print('=' * 72)
print('pattern1_conftest_hook/solution/solve.sh')
print('=' * 72)
with open('pattern1_conftest_hook/solution/solve.sh') as f:
    print(f.read())

In [ ]:
# Helper: shell out to a pinned venv's python, run _attack_runner.py, and
# parse the single JSON line it prints to stdout. We can't import two
# versions of benchflow in the same Python process, so each version runs
# in its own subprocess.

import json
import os
import subprocess
from pathlib import Path

HERE = Path('.').resolve()
VENVS = HERE / '.venvs'


def run_one(label: str, venv_subdir: str) -> dict:
    python = VENVS / venv_subdir / 'bin' / 'python'
    if not python.exists():
        raise FileNotFoundError(
            f'venv not found at {python}. Run `python run_comparison.py` first '
            f'to create .venvs/bf-0.2.0/ and .venvs/bf-head/.'
        )

    env = os.environ.copy()
    env['BENCHJACK_JOBS_DIR'] = str(HERE / '.jobs' / f'nb-{label}')
    env['BENCHJACK_TRIAL_NAME'] = f'notebook-{label}'

    proc = subprocess.run(
        [str(python), str(HERE / '_attack_runner.py')],
        env=env,
        capture_output=True,
        text=True,
    )

    for line in reversed(proc.stdout.strip().splitlines()):
        line = line.strip()
        if line.startswith('{'):
            try:
                payload = json.loads(line)
                payload['label'] = label
                return payload
            except json.JSONDecodeError:
                continue
    return {
        'label': label,
        'version': None,
        'reward': None,
        'error': f'no JSON on stdout (rc={proc.returncode})',
        'stderr_tail': proc.stderr[-1000:],
    }


print('helper loaded; ready to run against both pinned venvs')

In [ ]:
# Run 1: benchflow==0.2.0 from PyPI. Under 0.2.0 defaults the oracle
# solve.sh runs as root, /tests is writable, and the planted conftest.py
# ends up at /tests/conftest.py where pytest picks it up at verify time.

result_020 = run_one('0.2.0', 'bf-0.2.0')
result_020

### 0.2.0 result

Expected: `reward = 1.0`. The three `assert` statements in `test_outputs.py` never execute — pytest's collection step replaces every test body with a no-op before any of them run, so all three "pass" and the verifier writes `1.0` to `/logs/verifier/reward.txt`.

This is the baseline BenchJack-style bypass, originally demonstrated against SWE-bench Verified and ported here to the minimal skillsbench-shaped task in `attack_task/`.

In [ ]:
# Run 2: benchflow HEAD (editable install from this repo). Same task, same
# solve.sh, same exploit payload — but four independent hardening layers
# now stand between the attacker and the verifier's pytest run.

result_head = run_one('head', 'bf-head')
result_head

### HEAD result

Expected: `reward = 0.0`. Three independent defense layers each break the exploit; any one of them is sufficient.

1. **`/tests` is locked.** Sandbox setup runs `chown root:root /tests && chmod 700 /tests`. The primary drop in `solve.sh` (`cp $EXPLOIT /tests/conftest.py`) fails with permission denied. `solve.sh` falls through to its `/app/conftest.py` fallback.
2. **Pytest rootdir anchoring.** The verifier runs pytest with `--rootdir=/tests --confcutdir=/tests`. Even with a `conftest.py` sitting at `/app/conftest.py`, pytest's collection walk starts and ends inside `/tests` and never visits `/app`.
3. **Pre-verify cleanup.** Before the verifier runs, benchflow executes `find / -maxdepth 5 -name conftest.py -not -path '/tests/*' -delete`. Any stray `conftest.py` the agent managed to plant outside `/tests` is gone by the time pytest starts.

Note: `.pth` file injection (P7) is a separate attack — it is blocked by oracle running as `sandbox_user` (non-root), which denies writes to system site-packages. See `pattern7_pth_injection/` for that pattern.

See `docs/harden-sandbox.md` for the full seven-pattern audit.

---

## Audit your own benchmark in five minutes

If you maintain an agent benchmark that runs pytest in the agent's container, check these six things:

1. **Is your test directory writable by the agent user?** If yes, the agent can plant `conftest.py` there. Lock it with `chmod 700` owned by root.
2. **Does your verifier pin `--rootdir` and `--confcutdir`?** If pytest's rootdir walk includes any agent-writable path, a planted `conftest.py` can hook test collection.
3. **Do you clean up stray `conftest.py` files before verifying?** Defense in depth — even with the above two, a `find -delete` pass costs milliseconds.
4. **Does the agent run as non-root?** Running as root defeats filesystem permissions entirely. `setpriv --reuid=<user>` or `su -l <user>` in the sandbox entrypoint.
5. **Do you delete agent-written `.pth` and `sitecustomize.py` files before verifying?** Python's import system loads them at startup; they can register atexit handlers that force exit code 0 regardless of test results.
6. **Is `PATH` canonical when the verifier runs?** If agent-writable directories sit earlier on `PATH`, the agent can shim `pytest` with a fake binary.

Benchflow 0.2.1 does all six by default. For benchmarks not built on benchflow, the same mitigations apply regardless of the orchestrator you use.